# 06 · The answer, the board number, and the cold run

**Produces:** `slides/slide4_chart.png`, `slides/gap_by_city.png`, `slides/offer_window.png`, `slides/board_number.xlsx` (the one number with its denominator, period and comparison beside it), a row in `docs/cold-run-log.md`, and the headline printed in the **last cell**.

**Needs:** notebooks 03 and 05 run.

This notebook is short on purpose. It is the one the panel will ask us to run from a fresh Codespace (Kernel, Restart and Run All), so it reads only the clean table and does nothing clever. **The number printed in the last cell must be the number on slide 3.**

## CONFIG
Everything that depends on the machine or the period lives in this one cell, so nothing else needs editing.

In [ ]:
# ---------- imports ----------
import gzip
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# show more columns when we print a dataframe
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

# ---------- paths ----------
# the notebooks live in the notebooks folder, so the repo root is one folder up
if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent
else:
    ROOT = Path.cwd()

RAW = ROOT / "data" / "raw"       # the data pack as we received it (we never edit this)
CLEAN = ROOT / "data" / "clean"   # the files our notebooks write
SLIDES = ROOT / "slides"          # charts and the excel file for the slides
CLEAN.mkdir(parents=True, exist_ok=True)
SLIDES.mkdir(parents=True, exist_ok=True)

# ---------- the two periods we compare ----------
H1_2025 = ("2025-01-01", "2025-06-30 23:59:59")   # 23:59:59 so that 30 June is included
H1_2026 = ("2026-01-01", "2026-06-30 23:59:59")

# a small helper: prints PASS or FAIL but never stops the notebook
def check(name, condition, detail=""):
    if condition:
        print("PASS", name, detail)
    else:
        print("FAIL", name, detail)

print("repo root:", ROOT)

In [ ]:
notebook_start = time.time()      # for the runtime of the cold run

# ---------- TEAM SETTINGS: edit this cell ----------
BRIEF = "A"
GROUP = "09"
WHO_RAN_THIS = "<name of the member who did not write the notebooks>"

# False = the charts are only saved as png files in the slides folder and nothing is drawn on the screen (good for the live demo)
SHOW_CHARTS = False

# the headline from our last agreed run. if the number below is different, find out why before going home
EXPECTED_H1_2025 = 435430
EXPECTED_H1_2026 = 457230

# write the sentence for the board in our own words. leave it empty ("") to use the draft that this notebook builds from the numbers
TEAM_CLAIM = ""
TEAM_CAVEAT = ""

In [ ]:
# ---------- load the clean table ----------
df = pd.read_csv(CLEAN / "bookings_clean.csv.gz", parse_dates=["slot_at"],
                 dtype={"booking_id": "str", "person_id": "str"}, low_memory=False)
print(df.shape)
print(df.dtypes)

df["half"] = "other"
df.loc[df["slot_at"].between(H1_2025[0], H1_2025[1]), "half"] = "H1_2025"
df.loc[df["slot_at"].between(H1_2026[0], H1_2026[1]), "half"] = "H1_2026"
df["month"] = df["slot_at"].dt.strftime("%Y-%m")
df_h = df[df["half"] != "other"].copy()

# ---------- the headline ----------
n_2025 = int((df_h["half"] == "H1_2025").sum())
n_2026 = int((df_h["half"] == "H1_2026").sum())
growth = (n_2026 / n_2025 - 1) * 100

plan_wide = pd.read_csv(RAW / "plan" / "plan_2026_H1.csv")
plan_total = int(plan_wide.drop(columns=["city", "metric"]).sum().sum())
plan_growth = (plan_total / n_2025 - 1) * 100
short_tests = plan_total - n_2026
gap_points = short_tests / n_2025 * 100

print("H1 2025 tests:", f"{n_2025:,}")
print("H1 2026 tests:", f"{n_2026:,}")
print("growth:", f"{growth:+.2f}%")
print("plan:", f"{plan_total:,}", " plan growth:", f"{plan_growth:+.2f}%")
print("short of plan:", f"{short_tests:,}", "tests =", f"{gap_points:.1f}", "points of the H1 2025 base")

In [ ]:
# ---------- is it the number we agreed on? ----------
check("H1 2025 equals the agreed number", n_2025 == EXPECTED_H1_2025, str(n_2025) + " vs " + str(EXPECTED_H1_2025))
check("H1 2026 equals the agreed number", n_2026 == EXPECTED_H1_2026, str(n_2026) + " vs " + str(EXPECTED_H1_2026))
if n_2025 != EXPECTED_H1_2025 or n_2026 != EXPECTED_H1_2026:
    print("The totals are different from the agreed run. Do NOT present yet.")
    print("Find the reason (a rule in notebook 03? a different merge?), write it in docs/decisions-log.md, then update the numbers on the slides.")

## 1 · The numbers the charts need

In [ ]:
# growth by city and channel
t = df_h.groupby(["city", "channel", "half"]).size().unstack("half")
t["growth_pct"] = (t["H1_2026"] / t["H1_2025"] - 1) * 100
g_clinic = t.xs("clinic", level="channel")["growth_pct"]
g_home = t.xs("home", level="channel")["growth_pct"]

# city totals, the plan and the gap in points
city_tot = df_h.groupby(["city", "half"]).size().unstack("half")
plan_long = plan_wide.melt(id_vars=["city", "metric"], var_name="month", value_name="plan_2026")
plan_long["city"] = plan_long["city"].replace({"Bangalore": "Bengaluru"})
city_tot["plan_2026"] = plan_long.groupby("city")["plan_2026"].sum()
city_tot["growth_pct"] = (city_tot["H1_2026"] / city_tot["H1_2025"] - 1) * 100
city_tot["plan_growth_pct"] = (city_tot["plan_2026"] / city_tot["H1_2025"] - 1) * 100
city_tot["gap_points"] = (city_tot["H1_2026"] - city_tot["plan_2026"]) / n_2025 * 100
print(city_tot.round(1))
print("sum of gap points:", round(city_tot["gap_points"].sum(), 2))

# channels as a whole
ch = df_h.groupby(["channel", "half"]).size().unstack("half")
ch["growth_pct"] = (ch["H1_2026"] / ch["H1_2025"] - 1) * 100
ch["change_tests"] = ch["H1_2026"] - ch["H1_2025"]
print(ch.round(1))
check("clinic + home = network", int(ch["H1_2026"].sum()) == n_2026 and int(ch["H1_2025"].sum()) == n_2025)

clinic_below_plan_everywhere = bool((g_clinic < city_tot["plan_growth_pct"]).all())
print("clinic growth below the city's plan growth in every city:", clinic_below_plan_everywhere)

## 2 · The charts for the slides

In [ ]:
# ---------- chart 1 (slide 4): growth by city and channel, with the plan ----------
order = list(city_tot.sort_values("growth_pct", ascending=False).index)
x = np.arange(len(order))
width = 0.38

fig, ax = plt.subplots(figsize=(8, 4.5))
bars_clinic = ax.bar(x - width / 2, g_clinic[order], width, color="#D63A6A", label="Clinic")
bars_home = ax.bar(x + width / 2, g_home[order], width, color="#4E38AB", label="Home")
ax.scatter(x, city_tot.loc[order, "plan_growth_pct"], marker="D", color="#1A1440", zorder=3, label="Plan (city)")

# write the value above or below each bar
for i in range(len(order)):
    city = order[i]
    ax.text(i - width / 2, g_clinic[city] - 1.5, f"{g_clinic[city]:.0f}%", ha="center", va="top", fontsize=8)
    ax.text(i + width / 2, max(g_home[city], city_tot.loc[city, "plan_growth_pct"]) + 1.5, f"{g_home[city]:+.0f}%", ha="center", va="bottom", fontsize=8)

ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(order)
ax.set_ylabel("Tests, % change H1 2026 vs H1 2025")
ax.set_xlabel("Denominator: H1 2025 tests of each city and channel")
ax.set_title(f"Growth by city and channel (network {growth:+.1f}%, plan {plan_growth:+.1f}%)")
# leave head room at the top so the legend does not sit on the labels
top_value = max(g_home.max(), city_tot["plan_growth_pct"].max())
ax.set_ylim(g_clinic.min() - 6, top_value * 1.3)
ax.legend(loc="upper right", ncol=3, fontsize=8)
plt.tight_layout()
fig.savefig(SLIDES / "slide4_chart.png", dpi=200)
if SHOW_CHARTS:
    plt.show()
else:
    plt.close(fig)       # closed, so the notebook does not draw it

In [ ]:
# ---------- chart 2: where the gap sits, by city (points) ----------
s = city_tot["gap_points"].sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(s.index), list(s.values), color="#D63A6A")
for i in range(len(s)):
    # put the value inside the bar, so it never touches the city names
    ax.text(s.values[i] / 2, i, f"{s.values[i]:.1f}", ha="center", va="center", color="white", fontsize=9)
ax.set_xlabel("Points of the gap vs plan (divided by " + f"{n_2025:,}" + " tests in H1 2025)")
ax.set_title(f"The {gap_points:.1f}-point gap, by city (adds up to {city_tot['gap_points'].sum():.1f})")
plt.tight_layout()
fig.savefig(SLIDES / "gap_by_city.png", dpi=200)
if SHOW_CHARTS:
    plt.show()
else:
    plt.close(fig)       # closed, so the notebook does not draw it

In [ ]:
# ---------- chart 3: the free home collection window, month by month ----------
m_all = df.groupby("month").size()
m_clinic = df[df["channel"] == "clinic"].groupby("month").size()
m_home = df[df["channel"] == "home"].groupby("month").size()

months = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
yoy_all = []
yoy_clinic = []
yoy_home = []
for m in months:
    last_year = m.replace("2026", "2025")
    yoy_all.append((m_all[m] / m_all[last_year] - 1) * 100)
    yoy_clinic.append((m_clinic[m] / m_clinic[last_year] - 1) * 100)
    yoy_home.append((m_home[m] / m_home[last_year] - 1) * 100)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axvspan(-0.4, 2.4, color="#E9B04C", alpha=0.25)          # the offer months
ax.plot(months, yoy_home, marker="o", color="#4E38AB", label="Home")
ax.plot(months, yoy_all, marker="o", color="#1A1440", linewidth=2.5, label="Network")
ax.plot(months, yoy_clinic, marker="o", color="#D63A6A", label="Clinic")
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("Tests, % change vs the same month of 2025")
ax.set_title("Free home collection (shaded: January to March)")
ax.legend()
plt.tight_layout()
fig.savefig(SLIDES / "offer_window.png", dpi=200)
if SHOW_CHARTS:
    plt.show()
else:
    plt.close(fig)       # closed, so the notebook does not draw it

q1_clinic = (m_clinic[["2026-01", "2026-02", "2026-03"]].sum() / m_clinic[["2025-01", "2025-02", "2025-03"]].sum() - 1) * 100
q2_clinic = (m_clinic[["2026-04", "2026-05", "2026-06"]].sum() / m_clinic[["2025-04", "2025-05", "2025-06"]].sum() - 1) * 100
q1_home = (m_home[["2026-01", "2026-02", "2026-03"]].sum() / m_home[["2025-01", "2025-02", "2025-03"]].sum() - 1) * 100
q2_home = (m_home[["2026-04", "2026-05", "2026-06"]].sum() / m_home[["2025-04", "2025-05", "2025-06"]].sum() - 1) * 100
print("clinic: Q1", round(q1_clinic, 1), "%  Q2", round(q2_clinic, 1), "%     home: Q1", round(q1_home, 1), "%  Q2", round(q2_home, 1), "%")

## 3 · The sentence and the Excel file
Write the sentence in `TEAM_CLAIM` (cell 3) in our own words. If it is empty, this notebook builds a draft from the numbers, and prints a warning. **A draft is not our answer until every member can explain every word of it.**

In [ ]:
# ---------- the draft sentence, built from the numbers ----------
clinic_total_growth = ch.loc["clinic", "growth_pct"]
home_total_growth = ch.loc["home", "growth_pct"]

if clinic_total_growth < 0:
    clinic_phrase = "which fell " + f"{abs(clinic_total_growth):.1f}" + "%"
else:
    clinic_phrase = "which grew only " + f"{clinic_total_growth:.1f}" + "%"
everywhere = ""
if clinic_below_plan_everywhere:
    everywhere = ", and clinic is below plan in all six cities"

draft_claim = (f"Test volume grew {growth:.1f}% in H1 2026 against H1 2025 ({n_2026:,} vs {n_2025:,} tests, six cities) versus a plan of {plan_growth:.1f}%: "
               f"the {gap_points:.1f}-point shortfall sits in the clinic channel, {clinic_phrase} while home collection grew {home_total_growth:.1f}%{everywhere}.")
draft_caveat = (f"The free home collection offer (Rs 150 fee waived, Jan to Mar 2026) overlaps the period. Clinic growth was {q1_clinic:.1f}% in the offer quarter and "
                f"{q2_clinic:.1f}% after it, so part of the clinic fall may be patients switching to home. Who used the offer is not yet tested.")

if TEAM_CLAIM == "":
    CLAIM = draft_claim
    print("WARN: using the auto-drafted sentence. Rewrite it in TEAM_CLAIM in your own words.")
else:
    CLAIM = TEAM_CLAIM
if TEAM_CAVEAT == "":
    CAVEAT = draft_caveat
    print("WARN: using the auto-drafted caveat. Write your own in TEAM_CAVEAT.")
else:
    CAVEAT = TEAM_CAVEAT
print()
print(CLAIM)
print()
print(CAVEAT)

In [ ]:
# ---------- the board number, in Excel, with a live formula check ----------
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

wb = Workbook()
ws = wb.active
ws.title = "Board number"

rows = [
    ("Kalpa Health | Brief " + BRIEF + " | Group " + GROUP, ""),
    ("", ""),
    ("The number", f"{growth:+.1f}%"),
    ("What it counts", "tests (booking-test rows) from the bookings table, after the cleaning rules in docs/decisions-log.md"),
    ("Numerator", n_2026),
    ("Denominator", n_2025),
    ("Period", "H1 2026 against H1 2025, six cities"),
    ("Compared with", f"plan: {plan_growth:+.1f}%  ({-gap_points:.1f} point gap; {short_tests:,} tests short)"),
    ("The sentence", CLAIM),
    ("The caveat", CAVEAT),
    ("Formula check", "=ROUND((B5/B6-1)*100,1)"),
]
for r in rows:
    ws.append(list(r))

ws["A1"].font = Font(bold=True, size=14)
ws["B3"].font = Font(bold=True, size=24)
ws.column_dimensions["A"].width = 18
ws.column_dimensions["B"].width = 90
for row in ws.iter_rows(min_row=3, max_row=11):
    row[1].alignment = Alignment(wrap_text=True, vertical="top")

wb.save(SLIDES / "board_number.xlsx")
print("saved slides/board_number.xlsx. open it: the formula in B11 must show", round(growth, 1), "and agree with B3")

## 4 · The cold run
Run this notebook from a **fresh Codespace**, as a member who did not write the notebooks, with Kernel > Restart and Run All. The checks below must all say PASS. The row it writes to `docs/cold-run-log.md` is evidence for the panel. If the headline moved since the last run, nobody goes home until we know why.

In [ ]:
import subprocess

runtime_seconds = time.time() - notebook_start

# ---------- the checks ----------
check("the clean table exists", (CLEAN / "bookings_clean.csv.gz").exists())
check("the decisions log from notebook 03 exists", (CLEAN / "decisions_log_auto.csv").exists())
check("the column checks from notebook 02 exist", (CLEAN / "column_checks.csv").exists())
check("exactly six cities", df["city"].nunique() == 6, str(sorted(df["city"].unique())))
check("headline is a finite number", np.isfinite(growth))
check("headline equals the agreed number", n_2025 == EXPECTED_H1_2025 and n_2026 == EXPECTED_H1_2026)
check("this notebook ran in under 10 minutes (target T4)", runtime_seconds < 600, str(round(runtime_seconds)) + " seconds")

# ---------- the commit we ran on ----------
try:
    sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
except Exception:
    sha = "no git"

# ---------- one row in the cold run log ----------
docs_folder = ROOT / "docs"
docs_folder.mkdir(exist_ok=True)
log_path = docs_folder / "cold-run-log.md"

previous_headline = ""
if log_path.exists():
    for line in open(log_path).read().splitlines():
        if line.startswith("| 20"):
            previous_headline = line
this_headline = f"{growth:+.2f}%"
if previous_headline == "":
    same_as_last = "first run"
elif this_headline in previous_headline:
    same_as_last = "yes"
else:
    same_as_last = "NO: find out why before going home"

new_row = f"| {time.strftime('%Y-%m-%d')} | {WHO_RAN_THIS} | {round(runtime_seconds / 60, 1)} min | {this_headline} | {same_as_last} | {sha} |\n"
if not log_path.exists():
    open(log_path, "w").write("| date | who | runtime | headline | same as last | commit |\n|---|---|---|---|---|---|\n")
open(log_path, "a").write(new_row)
print("cold run row:", new_row.strip())

## The last cell
The number on slide 3 is the one printed below. If it is not, one of the two is wrong, and the panel will ask which.

In [ ]:
print(f"HEADLINE: {n_2026:,} of {n_2025:,} tests = {growth:+.1f}%  (H1 2026 vs H1 2025, six cities)")
print(f"PLAN:     {plan_total:,} tests = {plan_growth:+.1f}%  ->  {short_tests:,} tests short = {gap_points:.1f} points of the H1 2025 base")